## 0. ??????????????? ????????

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# ??????????? ?????????? ??? ????????????
import matplotlib.pyplot as plt
import seaborn as sns 
%matplotlib inline

# ????????? ??????????? ??????? ?????????? ??? ?????????? ????????:
from sklearn.model_selection import train_test_split

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

#import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
    #for filename in filenames:
        #print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install pycountry

In [ ]:
# ?????? ?????????? RANDOM_SEED, ????? ???? ???????????? ???? ??????????????!
RANDOM_SEED = 42

In [ ]:
# ??????????? ?????? ???????, ????? ???????????? ???? ??????????????:
!pip freeze > requirements.txt

In [ ]:
DATA_DIR = 'D:/Python/IDE/project_3/data'
df_train = pd.read_csv(DATA_DIR+'/hotels_train.csv') # ??????? ??? ????????
df_test = pd.read_csv(DATA_DIR+'/hotels_test.csv') # ??????? ??? ????????????
sample_submission = pd.read_csv(DATA_DIR+'/submission.csv') # ????????

In [ ]:
# ????????? ???? ?????? ?? ????????????

#DATA_DIR = '/kaggle/input/sf-booking/'
#df_train = pd.read_csv(DATA_DIR+'/hotels_train.csv') # ??????? ??? ????????
#df_test = pd.read_csv(DATA_DIR+'hotels_test.csv') # ??????? ??? ????????????
#sample_submission = pd.read_csv(DATA_DIR+'/submission.csv') # ????????

In [ ]:
df_train.info()

In [ ]:
df_train.head(2)

In [ ]:
df_test.info()

In [ ]:
df_test.head(2)

In [ ]:
sample_submission.head(2)

In [ ]:
sample_submission.info()

In [ ]:
# ?????! ??? ?????????? ????????? ????????? ?????????? ????? ? ???? ? ???? ???????
df_train['sample'] = 1 # ???????? ??? ? ??? ?????
df_test['sample'] = 0 # ???????? ??? ? ??? ????
df_test['reviewer_score'] = 0 # ? ????? ? ??? ??? ???????? reviewer_score, ?? ??? ?????? ???????????, ?? ????? ???? ?????? ????????? ??????

hotels = pd.concat([df_test, df_train], sort=False).reset_index(drop=True) # ??????????

In [ ]:
hotels.info()

## 1. ??????? ???????? ???????? ????????

In [ ]:
# ???? ??????? ??????? ? ?????????
cols_null_percent = hotels.isnull().mean()*100
cols_with_null = cols_null_percent[cols_null_percent > 0].sort_values(ascending=False)
print(cols_with_null)

In [ ]:
# ???????? ????????? ??????, ?? ?? ??? ???????? ? ?????
#hotels['lat'] = hotels['lat'].fillna(0)
#hotels['lng'] = hotels['lng'].fillna(0)

In [ ]:
# ????????, ????? ????? ????? ???????? ? ??????????? - ?????? ????, ????? ????????? ??????? ???????? ?????? ? ??????????
hotels[hotels['lat'].isnull()]['hotel_address'].value_counts()

In [ ]:
hotels[hotels['lng'].isnull()]['hotel_address'].value_counts()
# ?????????, ??? ???????? ?? ??? ?? ???????

In [ ]:
# ????????? ??????? ? ?????? ??????????
name_and_address = hotels[hotels['lat'].isnull()].groupby(['hotel_address', 'hotel_name']).count()[['lat', 'lng']]
name_and_address

In [ ]:
# ????? ?????? ?????? ?? ??????? ??? ?? ????? ?? ???????? ???????? ???????
hotels_null_coords = hotels[hotels['lat'].isnull()].groupby(['hotel_address']).count()[['lat', 'lng']]
hotels_null_coords

In [ ]:
hotels_coords = [[48.8399957, 2.3070823], 
           [48.8889127, 2.3309643], 
           [48.8753208, 2.3212033], 
           [41.391552, 2.1757053],
           [48.2208555, 16.3644228],
           [48.233592, 16.3434781],
           [48.2094054, 16.3512565],
           [48.2095525, 16.3492746],
           [48.1888183, 16.381591],
           [48.1868142, 16.418528],
           [41.3927888, 2.1673975],
           [48.219573, 16.2834034],
           [41.3832425, 2.1607609],
           [48.2458909, 16.3397395],
           [48.2135906, 16.3777195],
           [48.2168032, 16.357717],
           [48.220414, 16.3536299]]

In [ ]:
hotels = hotels.set_index('hotel_address')
hotels.update(hotels_null_coords)
hotels = hotels.reset_index()

In [ ]:
hotels.info()

In [ ]:
# ?????? ?????????? ????????, ???????????, ??? ? ??? ????? ?????????????? ?????????
hotels.nunique(dropna=False)

In [ ]:
# ???? ?????????
dupl_col=list(hotels.columns)
mask=hotels.duplicated(subset=dupl_col)
data_doubles=hotels[mask]
data_doubles.shape
# ? ??????? ??????? 336 ??????????. ??????, ?? ???????? ?????? ??????? ?????? ??????, ??? ??? ???? ??????? ??? ????.

## 2. ???????? ?? ?????????

In [ ]:
hotels.info()

### hotel_address

In [ ]:
# ???????? ? ??????? ????? (??????)
import pycountry
countries = {c.name for c in pycountry.countries} | {c.official_name for c in pycountry.countries if hasattr(c, "official_name")} | {c.alpha_3 for c in pycountry.countries} | {c.alpha_2 for c in pycountry.countries}

def extract_country(address):
    for country in countries:
        if address.endswith(country):
            return country
    return None

# ????????? ? ???????
hotels["country"] = hotels["hotel_address"].apply(extract_country)

In [ ]:
# ???????, ??? ??????????
hotels.country.value_counts()
# ?????????? ??????? ?????, ????????? ???????? ??????

In [ ]:
# ??????????? ????? ?? ??????
def extract_city(addr):
    parts = addr.split()
    if parts[-2:] == ['United', 'Kingdom']:
        return parts[-5]  # ??? UK ??????? ?????? ????? ??? ??? United Kingdom
    else:
        return parts[-2]  # ??? ???? ????????? ????? ????? ???????

# ??????? ??????? city
hotels["city"] = hotels["hotel_address"].apply(extract_city)


In [ ]:
hotels['city'].value_counts()

?????? ??? - ? ???????? ??? ????????? ?? ????? ?????? ????????? ? ????? ??????. ???? ?? ??????? - ?????????? ??? ?????? ??? ??????? ?????????? ?????????

### ???? ?????

In [ ]:
#??????????? ? datetime
hotels['review_date'] = pd.to_datetime(hotels['review_date'], dayfirst=True, errors='coerce')

# ??????? ???????? ???, ?????, ???? ?????? ? ???????? ?? ???? ????????. ???? ????? ???????? ?? ????????????? ?????????, ?????? ??? ????? ???? ??????
# ? ?????? ????????? (?????? - "???????" ???????? ???? ?????????? ? ?????? ????)
hotels['year'] = hotels['review_date'].dt.year
hotels['month'] = hotels['review_date'].dt.month
hotels['dayofweek'] = hotels['review_date'].dt.weekday  # 0 = ???????????, 6 = ???????????
hotels['is_weekend'] = (hotels['dayofweek'] >= 5).astype('int8')
hotels["quarter"] = hotels["review_date"].dt.quarter
hotels["weekofyear"] = hotels["review_date"].dt.isocalendar().week.astype("int16")

In [ ]:
#??????
def season(m):
    return ("winter" if m in [12,1,2] else
            "spring" if m in [3,4,5] else
            "summer" if m in [6,7,8] else "autumn")
hotels["season"] = hotels["month"].map(season)
# ????? ???????? ????? get_dummies
hotels = pd.get_dummies(hotels, columns=['season'], drop_first=True)

In [ ]:
# ???????? ??? ?? ??????????
hotels[['year', 'month', 'dayofweek', 'is_weekend']].head()

### hotel_name 

In [ ]:
# ??????? ????? ??????? ? ????? / ??????
hotels["hotel_reviews_cnt"] = hotels["hotel_name"].map(hotels["hotel_name"].value_counts())
hotels["city_reviews_cnt"]  = hotels["city"].map(hotels["city"].value_counts())

In [ ]:
hotels['hotel_name'].describe()

In [ ]:
# ????????? ???????????? ? ??????? frequency ? targetencoding
#from sklearn.model_selection import KFold

# freq
#freq = hotels['hotel_name'].value_counts()
#hotels['hotel_name_freq'] = hotels['hotel_name'].map(freq).astype('int32')

# target
#kf = KFold(n_splits=5, shuffle=True, random_state=42)
#te = np.zeros(len(hotels))
#for tr, va in kf.split(hotels):
  #  m = hotels.iloc[tr].groupby('hotel_name')['reviewer_score'].mean()
   # te[va] = hotels.iloc[va]['hotel_name'].map(m)
#hotels['hotel_name_te'] = te

In [ ]:
#???????????? - ???????? ??????????? ? ??????? category_encoders
#import category_encoders as ce
#bin_encoder = ce.BinaryEncoder(cols=['hotel_name'])
#data_bin=bin_encoder.fit_transform(hotels['hotel_name'])
#hotels = pd.concat([hotels, data_bin], axis=1)

In [ ]:
# ???????? ???????? ?????? ????? get_dummies - ????? ???????? ????????? ?????????
hotels = pd.get_dummies(hotels, columns=['hotel_name'], drop_first=True)

### reviewer_nationality

In [ ]:
# ??????? ??????? ? ??????????????
hotels["reviewer_nationality"] = hotels["reviewer_nationality"].str.strip()

In [ ]:
a=hotels['reviewer_nationality'].value_counts()
print(len(a))

In [ ]:
# ?????? ??????? ????, ??? ????? ???????? ????????????????? (?????? - ??, ??? ????? ?????? ?????? ????? ?????? ?????????? ? ?????)
hotels['is_domestic'] = (
    hotels['reviewer_nationality'].astype(str).str.strip().str.casefold()
    == hotels['country'].astype(str).str.strip().str.casefold()
).astype('int8')

In [ ]:
# ?????????????? ???????? - ??????? ?? ?????? ??? ????? ?????? ? ??????? ?? ?????? ??? ????????????
hotels["domestic_x_business"] = hotels["is_domestic"] * hotels.get("trip_business_trip", 0)
hotels["domestic_x_leisure"]  = hotels["is_domestic"] * hotels.get("trip_leisure_trip", 0)

In [ ]:
import numpy as np
from sklearn.model_selection import KFold

# Frequency encoding (?????? ?????????? ??????? ?? ??????)
freq = hotels['reviewer_nationality'].value_counts()
hotels['reviewer_nationality_freq'] = hotels['reviewer_nationality'].map(freq)

# Target encoding ? ??????? ?? NaN
kf = KFold(n_splits=5, shuffle=True, random_state=42)
te = np.zeros(len(hotels))

global_mean = hotels['reviewer_score'].mean()

for train_idx, val_idx in kf.split(hotels):
    means = hotels.iloc[train_idx].groupby('reviewer_nationality')['reviewer_score'].mean()
    # map + fillna ?????????? ???????
    te[val_idx] = hotels.iloc[val_idx]['reviewer_nationality'].map(means).fillna(global_mean)

hotels['reviewer_nationality_te'] = te

### negative_review/positive_review

In [ ]:
# ???????? ? ??????? ?????????????? ?????? negative_review ????? ntlk ? ?????? ??????? ????????? ???????
import re
import nltk
nltk.download('vader_lexicon')

from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def clean_text(s: str) -> str:
    if not isinstance(s, str): 
        return ""
    s = s.strip()
    s = re.sub(r"(https?://\S+)|(\S+@\S+)", " ", s)                     # ?????? URL ? email
    s = re.sub(r"[^A-Za-z0-9\s\.\,\!\?\-']", " ", s)                    # ??????? ?????? ???????? ???????
    s = re.sub(r"\s+", " ", s).strip()                                  # ?????? ?????? ???????
    return s

# ??????? ??????
hotels["neg_clean"] = hotels["negative_review"].map(clean_text)

# ????????
hotels["neg_len_chars"] = hotels["neg_clean"].str.len()
hotels["neg_len_words"] = hotels["neg_clean"].str.split().str.len()
hotels["exclam_cnt"]    = hotels["neg_clean"].str.count(r"!")
hotels["quest_cnt"]     = hotels["neg_clean"].str.count(r"\?")
hotels["caps_ratio"]    = hotels["negative_review"].map(
    lambda s: (sum(c.isupper() for c in s) / max(1, len(s))) if isinstance(s, str) else 0.0
)
hotels["neg_vader"] = hotels["neg_clean"].map(lambda s: sia.polarity_scores(s)["compound"])
#hotels["is_no_negative"] = hotels["neg_clean"].str.fullmatch(r"(?i)no negative").fillna(False).astype("int8") # ???? ????? ??????? ?? ????? "No Negative" ?? ?? ?????????????


In [ ]:
# ???? ?? ????? ???????????? ?? ??? ???????? ???? ??? ??????????? ?????? ??????, ???????? ????? ????????, ???? ?????????? ????? ?? ???? ???????????

# ????????????: ?????? ???????, ??????? ??? ????? ????/????/????????, ?????????? ???????
def _norm(s: str) -> str:
    s = s.lower().strip()
    # ?????? ????/????? ?? ?????? ??????
    s = re.sub(r"(https?://\S+)|(\S+@\S+)", " ", s)
    # ??????? ?????/?????/???????
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# ??????? ????? (???????????????)
STRICT_SET = {
    "no negative", "no negatives", "nothing negative",
    "nothing", "nothing really", "nothing at all", "nothing in particular",
    "no complaints",
    "nothing to dislike", "nothing to complain about", "nothing not to like",
    "absolutely nothing",
    "none", "nil", "na",  # ???? ?????? ? "n a"/"n/a" ????? ????????????
    "all good", "all was good", "nothing all good",
    "everything", "everything was good", "everything was great",
    "everything was perfect", "everything was fine",
    "liked everything", "i liked everything",
    "see above", "as above",
    "no thing",
}

# ??????????? (???????????????)
EXTENDED_SET = {
    "no",                  # ????? "no", ?? "no parking"
    "leaving", "having to leave",
    "not applicable",
    "non",                 # FR "non" (???? ?????? ?????? ??? ?????)
    "not much",
    "cant think of anything"  # ??????? ??? ?????????
}

# ????????? ?????? ??? "can't think of anything" ? ?????????? ? ?????????? ????????
CANT_THINK_RE = re.compile(r"\bcan\s*t\s*think\s*of\s*anything\b")

# ??????, ??????? ??????? ????????? ????? ?????? ????????
ONLY_DECOR_RE = re.compile(r"^\s*[\(\)\[\]\{\}\-\_\.\,\!\?\/\\\|~\*]+\s*$")

def is_no_negative_text(s: str, use_extended: bool = True) -> int:
    if not isinstance(s, str) or not s.strip():
        return 1  # ?????? ?????? ? no negative

    # ???? ???? ??????/????? ? ??????? ??????
    if ONLY_DECOR_RE.fullmatch(s):
        return 1

    ns = _norm(s)
    if ns == "":
        return 1

    # ????? ????????? ? ??????? ??????
    if ns in STRICT_SET:
        return 1

    # "can't think of anything" ? ?????? ???????????/?????????
    if CANT_THINK_RE.search(ns):
        return 1

    # ???????????
    if use_extended and ns in EXTENDED_SET:
        return 1

    return 0

# ?????????? ? ??????? (????? ???????????? ??? ????????? ?????, ????. hotels['neg_clean'])
hotels["is_no_negative"] = hotels["neg_clean"].map(lambda x: is_no_negative_text(x, use_extended=True)).astype("int8")


In [ ]:
# ????????? ????????????? ?????? - ????? ? ???? positive_review ?? ???? ??? ?????????????? ??????
FAKE_POSITIVE_PATTERNS = [
    r"^no\s+positive$",
    r"^nothing$",
    r"^none$",
    r"^nil$",
    r"^n\s*a$",
    r"^na$",
    r"^not\s+applicable$",
    r"^nothing\s+really$",
    r"^nothing\s+at\s+all$",
    r"^nothing\s+special$",
    r"^nothing\s+in\s+particular$",
    r"^absolutely\s+nothing$",
    r"^nothing\s+to\s+mention$",
    r"^nothing\s+worth\s+mentioning$",
    r"^nothing\s+comes\s+to\s+mind$",
    r"^nothing\s+good$",
    r"^nothing\s+positive$",
    r"^no\s+comment$",
    r"^not\s+much$",
    r"^not\s+really$",
    r"^can.?t\s+think\s+of\s+anything$",
    r"^can.?t\s+say\s+anything\s+positive$",
    r"^didn.?t\s+like\s+anything$",  
]

fake_pos_re = re.compile("|".join(FAKE_POSITIVE_PATTERNS), flags=re.I)

hotels["is_no_positive"] = hotels["positive_review"].map(
    lambda s: bool(fake_pos_re.fullmatch(str(s).strip())) if isinstance(s, str) else False
).astype("int8")


In [ ]:
sia = SentimentIntensityAnalyzer()

def clean_text(s: str) -> str:
    if not isinstance(s, str): 
        return ""
    s = s.strip()
    s = re.sub(r"(https?://\S+)|(\S+@\S+)", " ", s)
    s = re.sub(r"[^A-Za-z0-9\s\.\,\!\?\-']", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# ??????? ??????
hotels["pos_clean"] = hotels["positive_review"].map(clean_text)

# ????????
hotels["pos_len_chars"] = hotels["pos_clean"].str.len()
hotels["pos_len_words"] = hotels["pos_clean"].str.split().str.len()
hotels["pos_exclam_cnt"] = hotels["pos_clean"].str.count(r"!")
hotels["pos_quest_cnt"]  = hotels["pos_clean"].str.count(r"\?")
hotels["pos_caps_ratio"] = hotels["positive_review"].map(
    lambda s: (sum(c.isupper() for c in s) / max(1, len(s))) if isinstance(s, str) else 0.0
)
hotels["pos_vader"] = hotels["pos_clean"].map(lambda s: sia.polarity_scores(s)["compound"])
hotels["is_no_positive"] = hotels["pos_clean"].str.fullmatch(r"(?i)no positive").fillna(False).astype("int8") # ???? ????? ??????? ?? ????? "No Positive" ?? ?? ?????????????

In [ ]:
# ????????????? ? ????????????? ?????
neg_words = ("dirty","smell","noisy","rude","slow","cold","hot","broken","poor","bad", 'small', 'expensive', 'worst', 'hate', 'terrible', 'awful', 'disappointing', 'disappointed',)
pos_words = ("clean","friendly","quiet","fast","warm","great","excellent","amazing","good","nice", 'quite', 'love', 'loved', 'lovely', 'perfect','perfectly',
             'quiet','spacious','modern', 'again')

def count_any(text, vocab): 
    s = str(text).lower()
    return sum(w in s for w in vocab)

hotels["neg_kw_cnt"] = hotels["neg_clean"].map(lambda s: count_any(s, neg_words))
hotels["pos_kw_cnt"] = hotels["pos_clean"].map(lambda s: count_any(s, pos_words))

### TAGS

In [ ]:
hotels['tags'].value_counts()

??????? ?????????? ? dtale ??? ?? ???????????, ????? ? ???????? ???? ???? ? ???????? ??, ??????? ??????? ???????? ????????????

In [ ]:
# ??????????? ??????? 'tags' ?? ?????? ? ??????
hotels['tags'] = hotels['tags'].apply(eval)

In [ ]:
# ??????? ??????? ????? - ??????? ??????? ?? ????? ? ???????? ??????? ?????? ???? ?? _
def clean_tags(tag_list):
    return [tag.strip().replace(" ", "_") for tag in tag_list]

# ????????? ???????
hotels['tags'] = hotels['tags'].apply(clean_tags)

# ???????? ??? ???? ? ???? ??????? ??????
all_tags = [tag for tag_list in hotels['tags'] for tag in tag_list]

In [ ]:
from collections import Counter

tag_counts = Counter(all_tags)
print(tag_counts.most_common(100))

In [ ]:
import re
import ast

# ?? ?????? ??????: ???????? ??????? ? ???????
def to_list_safe(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            v = ast.literal_eval(x)
            return v if isinstance(v, list) else [x]
        except:
            return [x]
    return []  # None ? ??????

hotels["tags"] = hotels["tags"].apply(to_list_safe)

# ?????? ???????
hotels["tags"] = hotels["tags"].apply(lambda tags: [t.strip() for t in tags if isinstance(t, str)])

# ????? ????????? (??? ????? ????????)
def has_tag(tags, keyword):
    kw = keyword.lower()
    return int(any(kw in (t or "").lower() for t in tags))

# ???????? ????????
trip_types = ["Leisure trip", "Business trip"]
companions = ["Couple", "Solo traveler", "Family with young children",
              "Family with older children", "Group"]

for t in trip_types:
    hotels[f"trip_{t.replace(' ', '_').lower()}"] = hotels["tags"].apply(lambda x: has_tag(x, t))

for c in companions:
    hotels[f"companion_{c.replace(' ', '_').lower()}"] = hotels["tags"].apply(lambda x: has_tag(x, c))

# ??????'Stayed X night(s)' ????? regex
stay_re = re.compile(r"stayed\s+(\d+)\s+night", re.IGNORECASE)

def stay_length(tags):
    for t in tags:
        if not isinstance(t, str):
            continue
        m = stay_re.search(t)
        if m:
            try:
                num = int(m.group(1))
            except:
                continue
            if num == 1:
                return "short"
            elif num <= 3:
                return "medium"
            elif num <= 7:
                return "week"
            else:
                return "long"
    return None

hotels["stay_length"] = hotels["tags"].apply(stay_length)

# ??? ??????
room_keywords = ["Suite", "Apartment", "Studio", "Room"]

def room_type(tags):
    low = [(t or "").lower() for t in tags if isinstance(t, str)]
    for kw in room_keywords:
        if any(kw.lower() in t for t in low):
            return kw
    return None

hotels["room_type"] = hotels["tags"].apply(room_type)

# ???????? ???????
def has_kw(tags, kw): 
    return int(any(kw in t.lower() for t in tags))
amenities = ["breakfast", "parking", "spa", "gym", "pool", "airport", "metro", "city_center"]
for kw in amenities:
    hotels[f"tag_{kw}"] = hotels["tags"].apply(lambda ts: has_kw(ts, kw))


In [ ]:
# ???????? ????? ?????????? ? ????? ? ??? ??????
import category_encoders as ce
encoder = ce.OneHotEncoder(cols=["stay_length", "room_type"], use_cat_names=True)
hotels = encoder.fit_transform(hotels)  

### days_since_review

In [ ]:
# ???????? ? days_since_review
hotels["days_since_review"] = hotels["days_since_review"].str.extract(r'(\d+)').astype(int)

### ????????? ?????? ????

In [ ]:
# ?????????? ?? ?????? ??????
city_center = {
    "London": (51.5074, -0.1278),
    "Barcelona": (41.3851, 2.1734),
    "Paris": (48.8566, 2.3522),
    "Amsterdam": (52.3676, 4.9041),
    "Vienna": (48.2100, 16.3738),
    "Milan": (45.4642, 9.1900),
}

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlmb = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlmb/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

hotels["dist_center_km"] = hotels.apply(
    lambda r: haversine(r["lat"], r["lng"], *city_center.get(r["city"], (r["lat"], r["lng"]))), axis=1
)

## ???????? ???????

In [ ]:
hotels.info()

In [ ]:
# ???????? ??????
encoder = ce.OneHotEncoder(cols=["country"], use_cat_names=True)
hotels = encoder.fit_transform(hotels)  

In [ ]:
# ???????? ?????
encoder = ce.OneHotEncoder(cols=["city"], use_cat_names=True)
hotels = encoder.fit_transform(hotels)

### ??????? ??????

In [ ]:
#??????? ??? ??????? ? ????? object
hotels = hotels.drop(columns=[col for col in hotels.columns if hotels[col].dtype == 'object'])

In [ ]:
# ??????? ???? ??????
hotels = hotels.drop(columns=['review_date'])

In [ ]:
# ???????? ?????

# ???????? ?????? int64-???????
int64_cols = hotels.select_dtypes(include="int64").columns

# ????????? ?? ??? ???????? (?????? 0/1)
binary_cols = [col for col in int64_cols if hotels[col].dropna().nunique() == 2]

# ????????? ? int8
hotels = hotels.astype({col: "int8" for col in binary_cols})

In [ ]:
hotels.info()

# ????????????

In [ ]:
# ??????? ?????? ?????????????????? ?????????
# ????? ????? 0,75
#def high_corr_pairs(df, corr_thr=0.75):
   #corr = df.corr(numeric_only=True)  # ?????? ???????? ???????
   #pairs = []
   #for i in range(len(corr.columns)):
      #for j in range(i+1, len(corr.columns)):
         #if abs(corr.iloc[i, j]) >= corr_thr:
               #pairs.append((corr.columns[i], corr.columns[j], corr.iloc[i, j]))
   #return pairs


#pairs = high_corr_pairs(hotels, corr_thr=0.75)
#for a, b, c in pairs:
   #print(f"{a:20} ~ {b:20} corr={c:.3f}")

??? ?? ? ???????????? - ?????? ? ?????? ??????????? 1 ? 1. ??????? ??????.
???? ??????? ?????????? ????? ?????? ???? ? ????????. ??????? ????? ???????? - ???????, ??? ????? ?????????????.
??????? ????? ???? ????? ?????? ?.?. ??????? ?????????? ? ?????
???? ??????? ??????????? ????? ?????? ?????? ??? ?????? ? ???????? ????????? ????? ?????. ????????? ?????? ????? ?????????????? ???????, ?.?. ??? ????? ? ??? ????? ???????????. ???? ?????? ??????????? ? ???????? ????????? ??? ?? 0.831 - ??????? ???? ??????.

In [ ]:
hotels=hotels.drop(columns=['additional_number_of_scoring', 'review_total_negative_word_counts','review_total_negative_word_counts','review_total_positive_word_counts',
                           'review_total_positive_word_counts','days_since_review', 'country_Italy','country_Netherlands','country_Spain','country_United Kingdom',
                           'country_France','country_Austria','neg_len_chars','pos_len_chars', 'dayofweek'])

In [ ]:
hotels.info()

In [ ]:
# ?????? ??????? ???????? ?????
train_data = hotels.query('sample == 1').drop(['sample'], axis=1)
test_data = hotels.query('sample == 0').drop(['sample'], axis=1)

y = train_data.reviewer_score.values            # ??? ??????
X = train_data.drop(['reviewer_score'], axis=1)

In [ ]:
# ????????????? ??????????? ??????? train_test_split ??? ???????? ???????? ??????
# ??????? 20% ?????? ?? ????????? (???????? test_size)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

In [ ]:
# ?????????
test_data.shape, train_data.shape, X.shape, X_train.shape, X_test.shape

In [ ]:
# ??????????? ??????????? ??????????:
from sklearn.ensemble import RandomForestRegressor # ?????????? ??? ???????? ? ???????? ??????
from sklearn import metrics # ??????????? ??? ?????? ???????? ??????

In [ ]:
# ??????? ?????? (????????? ?? ???????)
model = RandomForestRegressor(n_estimators=100, verbose=1, n_jobs=-1, random_state=RANDOM_SEED)

In [ ]:
# ??????? ?????? ?? ???????? ?????? ??????
model.fit(X_train, y_train)

# ?????????? ????????? ?????? ??? ???????????? ???????? ?????????? ? ???????? ???????.
# ????????????? ???????? ?????????? ? ?????????? y_pred
y_pred = model.predict(X_test)

In [ ]:
# ?????????? ????????????? ???????? (y_pred) ? ????????? (y_test), ? ??????? ????????? ??? ? ??????? ??????????
# ??????? ?????????? Mean Absolute Error (MAE) ? ?????????? ??????? ?????????? ????????????? ???????? ?? ???????????.
print('MAPE:', metrics.mean_absolute_error(y_test, y_pred))

In [ ]:
# ? RandomForestRegressor ???? ??????????? ??????? ????? ?????? ???????? ??? ??????
plt.rcParams['figure.figsize'] = (10,10)
feat_importances = pd.Series(model.feature_importances_, index=X.columns)
feat_importances.nlargest(15).plot(kind='barh')

In [ ]:
test_data.sample(10)

In [ ]:
test_data = test_data.drop(['reviewer_score'], axis=1)

In [ ]:
sample_submission

In [ ]:
predict_submission = model.predict(test_data)

In [ ]:
predict_submission

In [ ]:
list(sample_submission)

In [ ]:
sample_submission['reviewer_score'] = predict_submission
sample_submission.to_csv('submission.csv', index=False)
sample_submission.head(10)